# Modelo: TextCF

- Embeddings: `all-MiniLM-L6-v2` (384 dimensiones)

In [1]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import sys
sys.path.append("..")
from src.evaluation import temporal_train_test_split, evaluate_model
from src.text_features import TextFeatureExtractor

MODEL_NAME = "textcf"
RESULTS_DIR = f"../results/{MODEL_NAME}"
N_FACTORS = 50


## Modelo

In [2]:
class TextCF:
    """
    Hibrido: combina la prediccion de SVD (filtrado colaborativo) con
    la similitud coseno entre el perfil textual del usuario y los
    embeddings de los Items.

    score(u,i) = alpha * norm(SVD_score(u,i)) + (1-alpha) * cosine(user_text_profile, text_emb_i)

    El perfil del usuario se construye como el promedio de los embeddings
    de los Item que valoran con rating >= like_threshold en train.
    """
    def __init__(self, text_embeddings, svd_model, alpha=0.5, like_threshold=3.0):
        self.alpha = alpha
        self._threshold = like_threshold
        self._svd = svd_model
        self._ids = list(text_embeddings.keys())
        self._matrix = np.stack([text_embeddings[b] for b in self._ids])
        self._id_to_idx = {bid: i for i, bid in enumerate(self._ids)}

    def _user_profile(self, user_id, train_reviews):
        liked = train_reviews[
            (train_reviews['user_id'] == user_id) &
            (train_reviews['stars'] >= self._threshold)
        ]['business_id'].tolist()
        liked = [b for b in liked if b in self._id_to_idx]
        if not liked:
            liked = train_reviews[train_reviews['user_id'] == user_id]['business_id'].tolist()
            liked = [b for b in liked if b in self._id_to_idx]
        if not liked:
            return None
        return np.mean([self._matrix[self._id_to_idx[b]] for b in liked], axis=0)

    def _svd_scores_normalized(self, user_id):
        svd = self._svd
        if user_id not in svd['user_index']:
            return {}
        u_idx = svd['user_index'][user_id]
        raw = svd['predicted'][u_idx]
        vmin, vmax = raw.min(), raw.max()
        denom = vmax - vmin if vmax > vmin else 1.0
        idx_to_bid = {v: k for k, v in svd['item_index'].items()}
        return {idx_to_bid[j]: float((raw[j] - vmin) / denom)
                for j in range(len(raw)) if j in idx_to_bid}

    def recommend(self, user_id, train_reviews, top_k=10):
        seen = set(train_reviews[train_reviews['user_id'] == user_id]['business_id'])
        cf_scores = self._svd_scores_normalized(user_id)
        profile = self._user_profile(user_id, train_reviews)
        if profile is not None:
            sims = cosine_similarity(profile.reshape(1, -1), self._matrix)[0]
            text_scores = {bid: float(sims[i]) for i, bid in enumerate(self._ids)}
        else:
            text_scores = {}
        final = {}
        for bid in self._ids:
            if bid in seen:
                continue
            final[bid] = self.alpha * cf_scores.get(bid, 0.0) + (1 - self.alpha) * text_scores.get(bid, 0.0)
        return sorted(final, key=final.get, reverse=True)[:top_k]

## Datos y embeddings

In [3]:
reviews = pd.read_csv('../data/processed/reviews.csv', parse_dates=['date'])
train_reviews, test_reviews = temporal_train_test_split(reviews, test_fraction=0.2)
text_embeddings = TextFeatureExtractor.load('text_embeddings.npz')
print(f'Train: {len(train_reviews)} | Test: {len(test_reviews)}')
print(f'Text embeddings: {len(text_embeddings)} items, dim={next(iter(text_embeddings.values())).shape[0]}')

Train: 83256 reviews | Test: 17191 reviews
Train: 83256 | Test: 17191
Text embeddings: 1151 items, dim=384


## Entrenar SVD base

In [4]:
users = train_reviews["user_id"].unique()
items = train_reviews["business_id"].unique()
user_idx = {u: i for i, u in enumerate(users)}
item_idx = {b: i for i, b in enumerate(items)}

rows = train_reviews["user_id"].map(user_idx)
cols = train_reviews["business_id"].map(item_idx)
vals = train_reviews["stars"].astype(float)
matrix = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)))

svd = TruncatedSVD(n_components=N_FACTORS, random_state=42)
U_sigma = svd.fit_transform(matrix)
Vt = svd.components_
predicted = U_sigma @ Vt

svd_model = {"predicted": predicted, "user_index": user_idx, "item_index": item_idx}
print(f"SVD: {len(users)} users x {len(items)} items, k={N_FACTORS}")
print(f"Explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}")


SVD: 10490 users x 1151 items, k=50
Explained variance ratio: 0.297


## Evaluacion completa

In [5]:
model = TextCF(text_embeddings, svd_model, alpha=0.5, like_threshold=4.0)
metrics = evaluate_model(
    lambda uid, top_k: model.recommend(uid, train_reviews, top_k),
    test_reviews, train_reviews, k_values=[5, 10, 20]
)
print(f'TextCF (alpha=0.5, like_threshold=4.0):')
print(metrics.round(4))

TextCF (alpha=0.5, like_threshold=4.0):
    precision  recall    ndcg
K                            
5      0.0213  0.0668  0.0487
10     0.0170  0.1049  0.0619
20     0.0129  0.1552  0.0762


## Guardar resultados

In [6]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics.to_csv(f'{RESULTS_DIR}/metrics.csv')
with open(f'{RESULTS_DIR}/config.json', 'w') as f:
    json.dump({'model': 'TextCF', 'alpha': 0.5, 'n_factors': N_FACTORS,
               'sbert_model': 'all-MiniLM-L6-v2', 'like_threshold': 4.0}, f, indent=2)
print(f'Saved -> results/{MODEL_NAME}/')

Saved -> results/textcf/
